##  Attention

In [1]:
import torch

q = torch.randn(3,4) # 3 words, each represented by 4 numbers
k = torch.randn(3,4)

print(q.shape)
print(k.transpose(-2,-1).shape) #rows become columns and vice versa

scores = q @ k.transpose(-2,-1) #@ is matrix multiplication
print(scores.shape)

torch.Size([3, 4])
torch.Size([4, 3])
torch.Size([3, 3])


In [2]:
import math

d_k = q.size(-1) # size of each vector
scores = scores/math.sqrt(d_k) #scaling by root(d_k)
weights = scores.softmax(dim=-1) # dim = -1 means along each row

print(weights)
print(weights.sum(dim=-1))

v = torch.randn(3,4)
output = weights @ v # matrix mulplication
print(output)
print(output.shape)


tensor([[0.2170, 0.2648, 0.5181],
        [0.6351, 0.2690, 0.0959],
        [0.3246, 0.3510, 0.3244]])
tensor([1.0000, 1.0000, 1.0000])
tensor([[1.0562, 0.9097, 1.0381, 0.4532],
        [0.6484, 1.4002, 0.5021, 0.5970],
        [0.8187, 0.8971, 0.7208, 0.4409]])
torch.Size([3, 4])


So here the second output adds up to 1 for each word after softmax.

Also the input and output shape remain same (3,4)

In [3]:
def attention(q,k,v):
  d_k = q.size(-1)
  scores = q @ k.transpose(-2,-1)/math.sqrt(d_k)
  weights = scores.softmax(dim=-1) # dim - row wise
  return weights @ v, weights

In [4]:
q = torch.randn(3,4) # 3 words each represented by 4 numbers
k = torch.randn(3,4)
v = torch.randn(3,4)

out, w = attention(q,k,v)
print(out.shape, w.shape)

torch.Size([3, 4]) torch.Size([3, 3])


In [5]:
print(w) #without masking

tensor([[0.3723, 0.4180, 0.2097],
        [0.4157, 0.3331, 0.2512],
        [0.1151, 0.1228, 0.7621]])


Masked self attention - so basically the 0's in mask converted to -1e9 as it is extremely negative value. This is done before softmax so that after application of softmax the value -1e9 changed to 0

In [6]:
def attention_mask(q,k,v,mask=None):
  d_k = q.size(-1)
  scores = q @ k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)  #mask
  weights = scores.softmax(dim=-1)
  return weights @ v, weights

In [7]:
q = torch.randn(3,4) # 3 words each represented by 4 numbers
k = torch.randn(3,4)
v = torch.randn(3,4)

mask = torch.tril(torch.ones(3,3))
print(mask)

out, w = attention_mask(q,k,v,mask)
print(w)
print(w.sum(dim=-1))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000],
        [0.5404, 0.4596, 0.0000],
        [0.3669, 0.2646, 0.3685]])
tensor([1., 1., 1.])


## Mutli head attention

(batch, heads, words, d_k) - Standard layout for multihead attention - which means (1 sentence, 2 heads, 3 words, 4 numbers per word)

In [8]:
x = torch.randn(1,3,8) # 1 sentence, 3 words, 8 numbers per word (batch, words, numbers)
print(x.shape)

h = 2
d_k = 8 // h

x = x.view(1,3,h,d_k)
print(x.shape)

x = x.transpose(1,2) #swaps the "words" and "heads" positions in the shape, so each head ends up with its own (words, numbers) table.
print(x.shape)

torch.Size([1, 3, 8])
torch.Size([1, 3, 2, 4])
torch.Size([1, 2, 3, 4])


In [9]:
# Let's test using same attention
q = torch.randn(1,2,3,4)
k = torch.randn(1,2,3,4)
v = torch.randn(1,2,3,4)

out,w = attention(q,k,v)
print(out.shape)
print(w.shape)

torch.Size([1, 2, 3, 4])
torch.Size([1, 2, 3, 3])


putting it together into MultiHeadedAttention

Now we need the pieces around this core:



1.   Split the input into heads (what you just did with view + transpose).
2.   Run your attention function.
3. Merge the heads back into one vector (transpose + view, reversed).
4. Linear layers: before splitting, the input passes through a learned nn.Linear for each of Q, K, V; after merging, one more nn.Linear at the end.

In [10]:
def split_heads(x,h):
  batch, words, d_model = x.shape
  d_k = d_model // h
  x = x.view(batch, words, h, d_k)
  x = x.transpose(1,2)
  return x

In [11]:
x = torch.randn(1,3,8)
x_new = split_heads(x,2)
print(x_new.shape)

torch.Size([1, 2, 3, 4])


In [12]:
# Let's test using same attention
q = torch.randn(1,2,3,4)
k = torch.randn(1,2,3,4)
v = torch.randn(1,2,3,4)

out,w = attention(q,k,v)
print(out.shape)
print(w.shape)

torch.Size([1, 2, 3, 4])
torch.Size([1, 2, 3, 3])


In [13]:
def merge_heads(x):
  batch, h, words, d_k = x.shape
  x = x.transpose(1,2)  # group by words instead of heads(shelves diagram remember)
  x = x.contiguous().view(batch,words, h*d_k) # merge heads back to original
  return x

In [14]:
x = torch.randn(1,2,3,4)
x_new = merge_heads(x)
print(x_new.shape)

torch.Size([1, 3, 8])


In [15]:
x = torch.arange(24).reshape(1, 2, 3, 4)
print("original (head, word, num):")
print(x)

y = x.transpose(1, 2).contiguous()
print("after transpose+contiguous (word, head, num):")
print(y)

merged = y.view(1, 3, 8)
print("merged (word, all 8 nums):")
print(merged)

original (head, word, num):
tensor([[[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11]],

         [[12, 13, 14, 15],
          [16, 17, 18, 19],
          [20, 21, 22, 23]]]])
after transpose+contiguous (word, head, num):
tensor([[[[ 0,  1,  2,  3],
          [12, 13, 14, 15]],

         [[ 4,  5,  6,  7],
          [16, 17, 18, 19]],

         [[ 8,  9, 10, 11],
          [20, 21, 22, 23]]]])
merged (word, all 8 nums):
tensor([[[ 0,  1,  2,  3, 12, 13, 14, 15],
         [ 4,  5,  6,  7, 16, 17, 18, 19],
         [ 8,  9, 10, 11, 20, 21, 22, 23]]])


## Full Multiheaded self attention class

word vectors (1, 3, 8)                                             
      |                     
   w_q(x)              <- ONE shared Linear(8,8), applied to full 8-number vectors         
      |                                                               
   (1, 3, 8)            <- still full-size, just transformed         
      |                                                             
   split_heads(..., h=2) <- NOW cut into 2 heads                 
      |                                                            
   (1, 2, 3, 4)          <- each head has its own 4 numbers per word

# add head dim so mask (B,T,T) -> (B,1,T,T) broadcasts over scores (B,H,T,T)

In [16]:
## Linear layers
import torch.nn as nn
class MultiheadedAttention(nn.Module):
  def __init__(self, h, d_model):
    super().__init__()
    self.h = h
    self.d_k = d_model // h # d_k = 8 // 2
    self.w_q = nn.Linear(d_model, d_model) #d_model = 8 # output = x @ W + b
    self.w_k = nn.Linear(d_model, d_model) # output = x @ W + b
    self.w_v = nn.Linear(d_model, d_model) # output = x @ W + b
    self.w_o = nn.Linear(d_model, d_model) # output = x @ W + b

  def forward(self, query, key, value, mask=None): # query = key = value = (1,3,8)
    if mask is not None:
        mask = mask.unsqueeze(1)          # (B, T, T) -> (B, 1, T, T), broadcasts over heads

    q = split_heads(self.w_q(query), self.h)
    k = split_heads(self.w_k(key), self.h)
    v = split_heads(self.w_v(value), self.h)

    out, weights = attention_mask(q,k,v,mask)

    out = merge_heads(out)
    return self.w_o(out)

In [17]:
mha = MultiheadedAttention(2,8)
x = torch.randn(1,3,8)

out = mha(x,x,x)
print(out.shape)

torch.Size([1, 3, 8])


## Position wise feed forward

FFN(x) = max(0, xW1 + b1)W2 + b2

In [18]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
      super().__init__()
      self.w1 = nn.Linear(d_model, d_ff) #callable so can be used like a function
      self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
      return self.w2((torch.relu(self.w1(x))))

In [19]:
pff = PositionWiseFeedForward(8,32)
x = torch.randn(1,3,8)
out = pff(x)
print(out.shape)

torch.Size([1, 3, 8])


## LayerNorm

In [20]:
class LayerNorm(nn.Module):
  def __init__(self, d_model, eps = 1e-6): # 8 = d_model, same 8 we've used throughout
    super().__init__()
    self.a_2 = nn.Parameter(torch.ones(d_model))
    self.b_2 = nn.Parameter(torch.zeros(d_model))
    self.eps = eps # small value added to denominator to prevent it from being zero incase std is 0 eg [3,3,3,3,3,3,3,3]

  def forward(self, x):
    mean = x.mean(-1, keepdim = True) #here -1 is last dimension which is 8, so 1 mean per word
    std = x.std(-1, keepdim = True)
    return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [21]:
ln = LayerNorm(8)
x = torch.randn(1,3,8) * 5 + 100

out = ln(x)

print(out.mean(-1))
print(out.std(-1))

tensor([[-1.5311e-06,  4.4145e-07,  9.0897e-07]], grad_fn=<MeanBackward1>)
tensor([[1.0000, 1.0000, 1.0000]], grad_fn=<StdBackward0>)


## Sublayer Connection : Add + Norm together

LayerNorm(x + Dropout(sublayer(x))                                   
sublayer can be attention or feed forward

In [22]:
class SublayerConnection(nn.Module):
  def __init__(self, d_model, dropout):
    super().__init__()
    self.norm = LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, sublayer):
    return self.norm(x + self.dropout(sublayer(x)))

In [23]:
mha = MultiheadedAttention(h=2,d_model=8)
x = torch.randn(1,3,8)

sb = SublayerConnection(d_model=8, dropout=0.1)
out = sb(x,lambda x: mha(x,x,x))
print(out.shape)

torch.Size([1, 3, 8])


## Quick pattern observed till now:                                   
1. MultiHeadedAttention(h, d_model): needs d_model to build w_q, w_k, w_v, w_o, each Linear(d_model, d_model), and to compute d_k = d_model // h.
2. PositionWiseFeedForward(d_model, d_ff): needs d_model for w1's input size and w2's output size, so the block still takes in and puts out d_model-sized vectors, even though the middle (d_ff) briefly expands.
3. LayerNorm(features): needs d_model (called features here) to size its a_2/b_2 parameter vectors, one scale and one shift per number in the word vector.
4. SublayerConnection(size, dropout): needs d_model (called size here) purely to pass along to the LayerNorm it creates internally.

## Encoder Layer

In [24]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, self_attn, feed_forward, dropout):
    super().__init__()
    self.d_model = d_model
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.sublayer1 = SublayerConnection(d_model, dropout)
    self.sublayer2 = SublayerConnection(d_model, dropout)

  def forward(self, x, mask):
    x = self.sublayer1(x, lambda x : self.self_attn(x,x,x,mask))
    x = self.sublayer2(x, self.feed_forward)
    return x

In [25]:
mha = MultiheadedAttention(h=2, d_model=8)
ffn = PositionWiseFeedForward(d_model=8, d_ff=32)
layer = EncoderLayer(d_model=8, self_attn=mha, feed_forward=ffn, dropout=0.1)

x = torch.rand(1,3,8)
mask = torch.ones(1,1,3)
out = layer(x,mask)
print(out)
print(out.shape)

tensor([[[-0.4900, -1.5735,  0.0088,  0.8419,  0.2456,  1.3006, -1.1068,
           0.7734],
         [-0.5248, -0.0495,  0.9991, -0.0704,  1.0862,  1.1007, -1.0485,
          -1.4928],
         [-0.6919, -0.8390, -0.0333,  0.4126,  2.0948, -0.1013, -1.0892,
           0.2473]]], grad_fn=<AddBackward0>)
torch.Size([1, 3, 8])


## Encoder - stacked encoder layers

In [26]:
import copy

def clones(module, N):
  return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [27]:
class Encoder(nn.Module):
  def __init__(self, layer, N):
    super().__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.d_model)

  def forward(self, x, mask):
    for l in self.layers:
      x = l(x, mask)
    return self.norm(x) # its the final layer norm added on top as final clean up (not to specific layer)

In [28]:
mha = MultiheadedAttention(h=2, d_model = 8)
ffn = PositionWiseFeedForward(d_model = 8, d_ff = 32)
one_layer = EncoderLayer(d_model = 8, self_attn = mha, feed_forward = ffn, dropout=0.1)

encoder = Encoder(one_layer, N=3)

x = torch.randn(1,3,8)
mask = torch.ones(1,1,3)
out = encoder(x, mask)
print(out.shape)

torch.Size([1, 3, 8])


## Embedding

1. Each integer ID got replaced by the 8-number row it points to. That's the entire mechanism: 3 IDs in, 3 rows out, each row d_model long.
2. x:            (1, 3)            batch=1, 3 token IDs (just integers, no "vector" dimension)
3. lut(x):       (1, 3, 8)      batch=1, 3 words, each now replaced by its 8-number row

In [29]:
class Embedding(nn.Module):
  def __init__(self, d_model, vocab_size):
    super().__init__()
    self.lut = nn.Embedding(vocab_size, d_model) # lut(x) is the look up table
    self.d_model = d_model

  def forward(self, x):
    return self.lut(x) * math.sqrt(self.d_model) #we multiply with sqrt(d_model) so that while applying the positional encoding the value should not be small

In [30]:
emb = Embedding(d_model = 8, vocab_size=11) # 11 is all possible words present in the vocabulary and we are searching for 1 sentence, asking for 3 IDs
x = torch.tensor([[1,2,3], [4,5,6]]) # changed input to 2 sentences
print(x.shape)

out = emb(x)
print(out.shape)

torch.Size([2, 3])
torch.Size([2, 3, 8])


Positional Encoding - Check!

In [31]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [32]:
pe = PositionalEncoding(d_model=4, dropout=0.0)
print(pe.pe[0, 0])   # position 0
print(pe.pe[0, 1])   # position 1
print(pe.pe[0, 2])   # position 2

tensor([0., 1., 0., 1.])
tensor([0.8415, 0.5403, 0.0100, 0.9999])
tensor([ 0.9093, -0.4161,  0.0200,  0.9998])


In [33]:
pe = PositionalEncoding(d_model=8, dropout=0.0)
emb = Embedding(d_model=8, vocab_size=11)

x = torch.tensor([[1, 2, 3]])
embedded = emb(x)
print(embedded.shape)   # (1, 3, 8)

out = pe(embedded)
print(out.shape)        # (1, 3, 8) - same shape, positions added in

torch.Size([1, 3, 8])
torch.Size([1, 3, 8])


### Decoder layer
decoder has three sub-layers instead of two — masked self-attention, then cross-attention (Q from decoder, K/V from encoder), then FFN.

In [34]:
class DecoderLayer(nn.Module):
  def __init__(self, d_model, self_attn, cross_attn, feed_forward, dropout):
    super().__init__()
    self.d_model = d_model
    self.self_attn = self_attn
    self.cross_attn = cross_attn
    self.feed_forward = feed_forward
    self.dropout = dropout
    self.sublayer1 = SublayerConnection(self.d_model, self.dropout)
    self.sublayer2 = SublayerConnection(self.d_model, self.dropout)
    self.sublayer3 = SublayerConnection(self.d_model, self.dropout)

  def forward(self, x, memory, src_mask, tgt_mask):
    x = self.sublayer1(x, lambda x : self.self_attn(x,x,x,tgt_mask)) # self attention
    x = self.sublayer2(x,lambda x: self.cross_attn(x,memory,memory,src_mask)) #Cross-attention means: query comes from the decoder (x), key and value come from the encoder (memory). That's the whole point of this sub-layer, letting the decoder look at the encoder's output
    x = self.sublayer3(x, self.feed_forward)
    return x

In [35]:
mha1 = MultiheadedAttention(h=2, d_model=8)
mha2 = MultiheadedAttention(h=2, d_model=8) # one we are adding to cross attention
ffn = PositionWiseFeedForward(d_model=8, d_ff=32)
dec_layer = DecoderLayer(d_model=8, self_attn=mha1, cross_attn=mha2, feed_forward=ffn, dropout=0.1)

x = torch.randn(1, 3, 8)          # decoder input (3 target words)
memory = torch.randn(1, 4, 8)     # encoder output (4 source words) - note: can differ in length!
tgt_mask = torch.tril(torch.ones(1, 3, 3))
src_mask = torch.ones(1, 1, 4) # this allows to make (1,2,3,4) == (batch, heads, tgt_words, src_words)

out = dec_layer(x, memory, src_mask, tgt_mask)
print(out.shape)   # expect (1, 3, 8)

torch.Size([1, 3, 8])


Important thing to note from above code:
I.  What actually happens inside DecoderLayer

1. Self-attention: the decoder's 3 words attend to each other (among themselves only). Still 3 words after this step, nothing about the encoder involved yet.
2. Cross-attention: now the decoder's 3 words (as queries) look at the encoder's 4 words (memory, as keys/values) to pull in relevant source information. Output is still 3 words, each one now enriched with information gathered from the 4 source words. The 4 never becomes part of the decoder's word count, it's just what the decoder is allowed to look at.
3. FFN: processes each of the 3 words independently. Still 3 words.

## Decoder

In [36]:
class Decoder(nn.Module):
  def __init__(self, layer, N):
    super().__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.d_model)

  def forward(self, x, memory, src_mask, tgt_mask):
    for l in self.layers:
      x = l(x, memory, src_mask, tgt_mask)
    return self.norm(x)

In [37]:
mha1 = MultiheadedAttention(h=2, d_model=8)
mha2 = MultiheadedAttention(h=2, d_model=8)
ffn = PositionWiseFeedForward(d_model=8, d_ff=32)

one_dec_layer = DecoderLayer(d_model=8, self_attn=mha1, cross_attn=mha2, feed_forward=ffn, dropout=0.1)

decoder = Decoder(one_dec_layer, N=3)

x = torch.randn(1,4,8) # changed from 4 words to 3 words which changes target mask
memory = torch.randn(1,3,8)
src_mask = torch.ones(1,1,3)
tgt_mask = torch.tril(torch.ones(1,4,4))

out = decoder(x,memory, src_mask, tgt_mask)
print(out.shape)

torch.Size([1, 4, 8])


### Generator:
Turning decoder's output into actual word probabilities over vocabulary

In [38]:
class Generator(nn.Module):
  def __init__(self, d_model, vocab_size):
    super().__init__()
    self.proj = nn.Linear(d_model, vocab_size) # y = W * x + b - this is what linear means

  def forward(self, x):
    return torch.log_softmax(self.proj(x), dim=-1) # this means we now want d_model = 8 to be equal to vocab_size=11 (toy example)

In [39]:
gen = Generator(d_model = 8, vocab_size = 11)
x = torch.randn(1,3,8)
out = gen(x)
print(out.shape)

print(out[0,0].exp().sum())

torch.Size([1, 3, 11])
tensor(1., grad_fn=<SumBackward0>)


### Encoder and Decoder

In [40]:
class EncoderDecoder(nn.Module):
  def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.generator = generator

  def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src), src_mask)

  def decode(self, memory, src_mask, tgt, tgt_mask): # it need src_mask as well
    return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

  def forward(self, src, tgt, src_mask, tgt_mask):
    memory = self.encode(src, src_mask)
    return self.decode(memory, src_mask, tgt, tgt_mask)

In [41]:
d_model, h, d_ff, N, dropout = 8, 2, 32, 3, 0.1
src_vocab, tgt_vocab = 11, 11

src_embed = nn.Sequential(Embedding(d_model, src_vocab), PositionalEncoding(d_model, dropout))
tgt_embed = nn.Sequential(Embedding(d_model, tgt_vocab), PositionalEncoding(d_model, dropout))

enc_layer = EncoderLayer(d_model, MultiheadedAttention(h, d_model), PositionWiseFeedForward(d_model, d_ff), dropout)
encoder = Encoder(enc_layer, N)

dec_layer = DecoderLayer(d_model, MultiheadedAttention(h, d_model), MultiheadedAttention(h, d_model), PositionWiseFeedForward(d_model, d_ff), dropout)
decoder = Decoder(dec_layer, N)

generator = Generator(d_model, tgt_vocab)

model = EncoderDecoder(encoder, decoder, src_embed, tgt_embed, generator)

In [42]:
src = torch.tensor([[1,2,3,4]])         # 1 sentence, 4 source tokens
tgt = torch.tensor([[1,2,3]])           # 1 sentence, 3 target tokens (so far)
src_mask = torch.ones(1,1,4)
tgt_mask = torch.tril(torch.ones(1,3,3))

out = model(src, tgt, src_mask, tgt_mask)
print(out.shape)          # expect (1, 3, 8) - decoder output, still d_model

probs = model.generator(out)
print(probs.shape)        # expect (1, 3, 11) - word probabilities
print(probs[0,0].exp().sum())   # should be ~1.0

torch.Size([1, 3, 8])
torch.Size([1, 3, 11])
tensor(1., grad_fn=<SumBackward0>)


## Copy task to see whether the transformer really works

In [43]:
def data_gen(vocab_size, batch_size, n_batches):
    for _ in range(n_batches):
        data = torch.randint(1, vocab_size, size=(batch_size, 8))
        data[:, 0] = 1                      # force first token to a fixed "start" id
        src = data.clone()
        tgt = data.clone()
        yield src, tgt

In [44]:
gen = data_gen(vocab_size=11, batch_size=4, n_batches=1)
src, tgt = next(gen)
print(src)
print(tgt)

tensor([[ 1,  8,  3,  4,  6,  6,  1,  9],
        [ 1,  2,  1,  8,  8,  4,  8,  1],
        [ 1,  7, 10,  1,  7,  8, 10, 10],
        [ 1,  8,  5,  3, 10,  4,  4,  1]])
tensor([[ 1,  8,  3,  4,  6,  6,  1,  9],
        [ 1,  2,  1,  8,  8,  4,  8,  1],
        [ 1,  7, 10,  1,  7,  8, 10, 10],
        [ 1,  8,  5,  3, 10,  4,  4,  1]])


In [45]:
def make_masks(src, tgt, pad=0):
    src_mask = (src != pad).unsqueeze(-2)                       # (batch, 1, src_len)
    tgt_mask = (tgt != pad).unsqueeze(-2)                        # (batch, 1, tgt_len)
    causal = torch.tril(torch.ones(tgt.size(-1), tgt.size(-1))).bool()
    tgt_mask = tgt_mask & causal
    return src_mask, tgt_mask

In [46]:
criterion = nn.KLDivLoss(reduction="sum")

def label_smoothed_target(target, vocab_size, smoothing=0.1, pad=0):
    conf = 1.0 - smoothing
    dist = torch.full((target.size(0), vocab_size), smoothing / (vocab_size - 2))
    dist.scatter_(1, target.unsqueeze(1), conf)
    dist[:, pad] = 0
    return dist

In [47]:
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

def rate(step, d_model, warmup=400):
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: rate(step, d_model))

In [48]:
def train_epoch(model, data_iter, vocab_size):
    model.train()
    total_loss = 0
    for src, tgt in data_iter:
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        src_mask, tgt_mask = make_masks(src, tgt_in)

        out = model(src, tgt_in, src_mask, tgt_mask)
        log_probs = model.generator(out)

        smoothed = label_smoothed_target(tgt_out.reshape(-1), vocab_size)
        loss = criterion(log_probs.reshape(-1, vocab_size), smoothed)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    return total_loss

# add head dim so mask (B,T,T) -> (B,1,T,T) broadcasts over scores (B,H,T,T)

this was a mistake in Multihead Attention.

In [49]:
for epoch in range(300):
    loss = train_epoch(model, data_gen(vocab_size=11, batch_size=16, n_batches=20), vocab_size=11)
    print(f"epoch {epoch}, loss {loss:.1f}")

epoch 0, loss 4185.2
epoch 1, loss 4044.0
epoch 2, loss 3941.7
epoch 3, loss 3891.8
epoch 4, loss 3840.9
epoch 5, loss 3831.1
epoch 6, loss 3796.9
epoch 7, loss 3715.3
epoch 8, loss 3712.7
epoch 9, loss 3620.8
epoch 10, loss 3612.8
epoch 11, loss 3490.0
epoch 12, loss 3421.8
epoch 13, loss 3353.2
epoch 14, loss 3349.4
epoch 15, loss 3307.1
epoch 16, loss 3306.5
epoch 17, loss 3273.2
epoch 18, loss 3265.3
epoch 19, loss 3238.7
epoch 20, loss 3248.5
epoch 21, loss 3173.8
epoch 22, loss 3139.5
epoch 23, loss 3135.7
epoch 24, loss 3108.1
epoch 25, loss 3098.2
epoch 26, loss 3103.8
epoch 27, loss 3087.8
epoch 28, loss 3047.3
epoch 29, loss 3057.0
epoch 30, loss 3091.2
epoch 31, loss 3076.4
epoch 32, loss 3039.0
epoch 33, loss 2968.0
epoch 34, loss 2992.1
epoch 35, loss 2989.6
epoch 36, loss 3062.2
epoch 37, loss 3028.6
epoch 38, loss 3029.6
epoch 39, loss 2978.8
epoch 40, loss 2967.0
epoch 41, loss 3000.6
epoch 42, loss 2948.0
epoch 43, loss 2980.3
epoch 44, loss 2917.6
epoch 45, loss 2938.

In [50]:
@torch.no_grad()
def greedy_decode(model, src, max_len, start_symbol=1):
    model.eval()
    ys = torch.full((src.size(0), 1), start_symbol, dtype=torch.long, device=src.device)
    for _ in range(max_len - 1):
        src_mask, tgt_mask = make_masks(src, ys)
        out = model(src, ys, src_mask, tgt_mask)
        next_tok = model.generator(out[:, -1]).argmax(-1, keepdim=True)
        ys = torch.cat([ys, next_tok], dim=1)
    return ys

# Test on fresh data the model hasn't seen
src, tgt = next(data_gen(vocab_size=11, batch_size=5, n_batches=1))
pred = greedy_decode(model, src, max_len=tgt.size(1))

for i in range(5):
    print("src :", src[i].tolist())
    print("tgt :", tgt[i].tolist())
    print("pred:", pred[i].tolist())
    print()

src : [1, 9, 8, 3, 5, 8, 3, 6]
tgt : [1, 9, 8, 3, 5, 8, 3, 6]
pred: [1, 9, 3, 5, 3, 8, 6, 3]

src : [1, 10, 7, 4, 2, 3, 5, 1]
tgt : [1, 10, 7, 4, 2, 3, 5, 1]
pred: [1, 10, 7, 4, 5, 3, 1, 5]

src : [1, 2, 2, 8, 7, 6, 4, 6]
tgt : [1, 2, 2, 8, 7, 6, 4, 6]
pred: [1, 2, 2, 7, 8, 4, 6, 6]

src : [1, 1, 5, 10, 5, 6, 10, 10]
tgt : [1, 1, 5, 10, 5, 6, 10, 10]
pred: [1, 1, 5, 10, 5, 10, 6, 10]

src : [1, 1, 5, 2, 2, 7, 9, 10]
tgt : [1, 1, 5, 2, 2, 7, 9, 10]
pred: [1, 1, 5, 2, 2, 7, 9, 10]



The model has successfully learned to copy